In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

import joblib

sns.set_style("whitegrid")
SEED = 42
np.random.seed(SEED)

In [ ]:
def find_telco_csv():
    candidates = []
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower().endswith(".csv") and "telco" in f.lower():
                candidates.append(os.path.join(root, f))
    if not candidates:
        raise FileNotFoundError("Telco CSV not found under /kaggle/input")
    return sorted(candidates)[0]

CSV_PATH = find_telco_csv()
print("Using file:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
)

print("Shape:", df.shape)
display(df.head())

In [ ]:
print(df.info())
display(df.describe(include="all"))
print(df.isnull().sum())

In [ ]:
df.columns = [c.strip() for c in df.columns]

# Convert TotalCharges to numeric
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Target encoding
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop customerID because it is just an identifier
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

print("After cleaning shape:", df.shape)
display(df.head())
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x="Churn", data=df)
plt.title("Churn Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(df["tenure"], bins=30, kde=True)
plt.title("Tenure Distribution")
plt.show()

plt.figure(figsize=(8,5))
sns.boxplot(x="Churn", y="tenure", data=df)
plt.title("Tenure vs Churn")
plt.show()

plt.figure(figsize=(10,8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
log_reg_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED))
])

log_reg_param_grid = {
    "model__C": [0.1, 1.0, 10.0],
    "model__solver": ["liblinear", "lbfgs"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

log_reg_grid = GridSearchCV(
    log_reg_pipe,
    param_grid=log_reg_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

log_reg_grid.fit(X_train, y_train)

print("Best Logistic Regression Params:", log_reg_grid.best_params_)
print("Best CV F1:", log_reg_grid.best_score_)

In [ ]:
rf_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=SEED))
])

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    rf_pipe,
    param_grid=rf_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Params:", rf_grid.best_params_)
print("Best CV F1:", rf_grid.best_score_)

In [ ]:
best_models = {
    "Logistic Regression": log_reg_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_
}

results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df)

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = best_models[best_name]

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Best Model:", best_name)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"{best_name} (AUC = {roc_auc_score(y_test, y_proba):.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
if best_name == "Random Forest":
    rf_estimator = best_model.named_steps["model"]
    ohe_features = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
    all_features = np.concatenate([numeric_features, ohe_features])

    importances = pd.DataFrame({
        "feature": all_features,
        "importance": rf_estimator.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importances.head(15))

    plt.figure(figsize=(8,6))
    sns.barplot(data=importances.head(10), x="importance", y="feature")
    plt.title("Top Feature Importances")
    plt.show()

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/telco_churn_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, OUTPUT_DIR / "telco_churn_pipeline.joblib")
print("Saved pipeline to:", OUTPUT_DIR / "telco_churn_pipeline.joblib")

In [ ]:
print("Task completed successfully.")
print("Best model:", best_name)
print("Saved artifact:", OUTPUT_DIR / "telco_churn_pipeline.joblib")